# VRDet: train / fine-tune OBB trên Colab

Notebook train kiến trúc **VRDet** (detector OBB mới, không dùng Ultralytics) trên dataset có sẵn trong Google Drive.

- Nhãn theo chuẩn OBB 8 điểm: `class x1 y1 x2 y2 x3 y3 x4 y4`, chuẩn hoá [0, 1], kèm `data.yaml`. Dùng thẳng zip đang có, không cần chuyển đổi.
- Checkpoint ghi thẳng vào Drive mỗi epoch. Mất phiên thì chạy lại các ô từ đầu, train **tự resume**.
- GPU: *Runtime → Change runtime type*. **G4** (RTX PRO 6000) rẻ nhất trên mỗi ảnh. A100 hoặc L4 cũng được: batch tự chỉnh theo VRAM.

Chạy lần lượt từng ô. Chỉ cần sửa ô **2. Cấu hình**.

## 1. Mount Drive và cài VRDet

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Cài thư viện VRDet từ GitHub (luôn lấy bản mới nhất). Các thư viện phụ thuộc đã có sẵn trên Colab.
!pip install -q --force-reinstall --no-deps git+https://github.com/Nguyenchitrai62/vrdet.git
!nvidia-smi --query-gpu=name,memory.total --format=csv

import vrdet
print("VRDet", vrdet.__version__)

## 2. Cấu hình

- `DATASET`: `"floor_plan"` hoặc `"wall_color"` (hai dataset trong Drive), hoặc `"custom"` (tự điền `CUSTOM_ZIP` và `CUSTOM_YAML`).
- `MODEL`:
  - `"s"`, `"m"`, `"l"`, `"x"` để train từ đầu. Khởi tạo từ COCO, giấy phép Apache-2.0.
  - Đường dẫn `best.pt` trên Drive để **fine-tune** tiếp từ model đã train. Class trùng tên giữ lại trọng số, class mới học từ đầu.
- `SCALE`:
  - `"auto"`: trang nhỏ (dưới khoảng 1.6 lần `IMGSZ`) được thu về đúng 1 tile, giống YOLO resize. Trang lớn giữ độ phân giải gốc và cắt tile, để giữ nét mảnh.
  - Hoặc đặt số cụ thể, ví dụ `0.8`.

In [ ]:
DATASET  = "floor_plan"        # "floor_plan" | "wall_color" | "custom"
MODEL    = "s"                 # "s" | "m" | "l" | "x" | "/content/drive/MyDrive/Train/VRDet/<run>/best.pt"
RUN_NAME = "vrdet_s_floorplan"
EPOCHS   = 24                  # floor_plan (16k trang): 24 epoch bản S khoảng 2 giờ trên G4
IMGSZ    = 1024                # kích thước tile (bội số của 32)
SCALE    = "auto"              # "auto" hoặc số, ví dụ 0.8
BATCH    = None                # None = tự chọn theo VRAM
DRIVE_RUNS = "/content/drive/MyDrive/Train/VRDet"   # kết quả + checkpoint (resume được)

# dataset custom: zip trên Drive và đường dẫn data.yaml sau khi giải nén
CUSTOM_ZIP  = "/content/drive/MyDrive/Dataset/<ten>/<dataset>.zip"
CUSTOM_YAML = "/content/dataset_custom/<dataset>/data.yaml"

## 3. Giải nén dataset từ Drive

In [ ]:
import os, zipfile, glob, random
import cv2, numpy as np

def unzip(zip_path, dst):
    if not os.path.exists(zip_path):
        raise FileNotFoundError(zip_path)
    marker = os.path.join(dst, ".unzipped_" + os.path.basename(zip_path))
    if not os.path.exists(marker):                      # chỉ giải nén một lần mỗi phiên
        with zipfile.ZipFile(zip_path) as z:
            z.extractall(dst)
        open(marker, "w").close()
    print("OK:", zip_path, "->", dst)

if DATASET == "floor_plan":
    unzip("/content/drive/MyDrive/Dataset/Floor_plan/dataset_obb_train_v3.zip", "/content/dataset_local")
    DATA = "/content/dataset_local/dataset_obb_train_v3/data.yaml"   # train: ../train/images, val: ../valid/images
elif DATASET == "wall_color":
    unzip("/content/drive/MyDrive/Dataset/tu_ph/Wall_Color/data/dataset_wall_color_v3.zip", "/content")
    DATA = "/content/dataset_wall_color/data.yaml"
    with open(DATA, "w") as f:
        f.write("""path: /content/dataset_wall_color
train: images/train
val: images/val
names:
  0: wall
  1: note
  2: door
  3: slide_door
  4: double_door
  5: wall_300
  6: junction
""")
else:
    unzip(CUSTOM_ZIP, "/content/dataset_custom")
    DATA = CUSTOM_YAML
print(open(DATA).read())

# Kích thước trang -> SCALE
from vrdet.data.prepare import find_splits
_, splits = find_splits(DATA)
imgs = sorted(glob.glob(str(splits["train"] / "*")))
random.seed(0)
sides = [max(cv2.imread(p).shape[:2]) for p in random.sample(imgs, min(40, len(imgs)))]
med = float(np.median(sides))
if SCALE == "auto":
    SCALE = round(IMGSZ / med, 4) if IMGSZ < med <= 1.6 * IMGSZ else 1.0
tiles = int(np.ceil(max(med * SCALE - 200, 1) / (IMGSZ - 200))) ** 2 if med * SCALE > IMGSZ else 1
print(f"{len(imgs)} trang train, cạnh dài trung vị {med:.0f}px -> SCALE={SCALE}, khoảng {tiles} tile/trang")
print("val:", splits["val"] or "không có -> tự tách 15% train")

## 4. Train

- Lần đầu sẽ cắt tile vào `/root/.cache/vrdet` (vài phút), các lần sau dùng lại.
- Công thức mặc định là công thức tốt nhất đã đo: RFS, nhóm query một-nhiều, AQD, loss góc, matching theo IoU, 900 query khi suy luận.
- Mất phiên giữa chừng: chạy lại ô 1, 3 rồi ô này, train tiếp từ epoch cuối trên Drive. Muốn train lại từ đầu thì đổi `RUN_NAME`.

Lệnh tương đương bằng CLI:

```
!vrdet train data=<data.yaml> model=s epochs=24 imgsz=1024 scale=0.8 project=/content/drive/MyDrive/Train/VRDet name=<run>
```

In [ ]:
from vrdet import Detector

model = Detector(MODEL)
r = model.train(
    data=DATA,
    epochs=EPOCHS,
    imgsz=IMGSZ,
    scale=SCALE,
    batch=BATCH,
    project=DRIVE_RUNS,
    name=RUN_NAME,
    workers=8,
)
print("Kết quả:", r.save_dir)
print("best.pt:", r.best)

## 5. Kết quả: mAP theo class và đường loss

In [ ]:
import json, matplotlib.pyplot as plt

print(open(os.path.join(r.save_dir, "eval_val.txt")).read())
q900 = os.path.join(r.save_dir, "eval_val_q900.txt")
if os.path.exists(q900):
    print("--- 900 queries ---")
    print(open(q900).read())

rows = [json.loads(l) for l in open(os.path.join(r.save_dir, "metrics.jsonl")) if '"epoch"' in l]
ep = [x["epoch"] for x in rows]
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
ax[0].plot(ep, [x["total"] for x in rows]); ax[0].set_title("train loss"); ax[0].set_xlabel("epoch")
ev = [x for x in rows if "sub_mAP50" in x]
ax[1].plot([x["epoch"] for x in ev], [x["sub_mAP50"] for x in ev], "o-", label="mAP50")
ax[1].plot([x["epoch"] for x in ev], [x["sub_mAP50_95"] for x in ev], "o-", label="mAP50:95")
ax[1].set_title("val"); ax[1].legend(); ax[1].set_xlabel("epoch")
plt.show()

## 6. Dự đoán thử trên vài trang val

- Kết quả ghi vào `<run>/predict`.
- Mỗi trang ra 3 file:
  - `.txt`: `class x1 y1 ... x4 y4 score`, chuẩn hoá theo trang.
  - `.json`: toạ độ pixel và tên class.
  - `_vis.jpg`: ảnh có vẽ box.

In [ ]:
from IPython.display import Image, display

val_dir = splits["val"] or splits["train"]
samples = sorted(glob.glob(str(val_dir / "*")))[:3]
pred_dir = os.path.join(r.save_dir, "predict")
for p in samples:
    dets = model.predict(p, conf=0.3, save_dir=pred_dir)
    stem = os.path.splitext(os.path.basename(p))[0]
    display(Image(os.path.join(pred_dir, stem + "_vis.jpg"), width=900))

## 7. (Tuỳ chọn) Fine-tune từ model đã train sang dataset khác

Đặt `MODEL` ở ô 2 bằng đường dẫn `best.pt`, đổi `DATASET` và `RUN_NAME`, rồi chạy lại ô 3 và 4. Hoặc chạy trực tiếp như sau:

```python
ft = Detector("/content/drive/MyDrive/Train/VRDet/vrdet_s_floorplan/best.pt")
ft.train(data=DATA, epochs=30, imgsz=IMGSZ, scale=SCALE, project=DRIVE_RUNS, name="ft_wall_color")
```

- Kiến trúc và cỡ model lấy theo checkpoint.
- Class trùng tên giữ lại trọng số đã học.

## 8. Đánh giá lại hoặc suy luận một thư mục trang

```python
Detector(r.best).val(DATA)                                              # mAP chuẩn DOTA trên tập val
Detector(r.best).predict("/content/my_pages", conf=0.3, save_dir="/content/preds")
```

Hoặc dùng CLI: `!vrdet predict model=<best.pt> source=<thư mục ảnh> conf=0.3 save_dir=/content/preds`.